In [1]:
import numpy as np
import pandas as pd
import snowflake.connector
import requests
import io

In [2]:
#Conectar a snowflake con las credenciales
snowflake_connection = snowflake.connector.connect(user='esteban.correa@RAPPI.COM', 
                                   authenticator='externalbrowser', 
                                   account='hg51401', 
                                   warehouse="RP_PERSONALUSER_WH",
                                   database="FIVETRAN")

In [3]:
#Definir variable para la query de forecast
query_forecast = """
SELECT
    f.warehouse_id,
    w.store,
    f.retail_id,
    w.name,
    f.forecast,
    f.sales_units,
    f.date, 
    w.provider_name, 
    w.cat2_name
FROM fivetran.cpgs_turbo_ds_public.global_forecast_main f
INNER JOIN (
    SELECT DISTINCT 
        store,
        city_name,
        warehouse_id,
        provider_name,
        retail_id,
        CAST(vivo_retail_id AS NUMERIC) AS vivo_retail_id,
        cat2_name,
        name
    FROM fivetran.cpgs_cargo.es_finance_co_margins_history_long 
    WHERE created_at >= DATE_TRUNC('MONTH', CURRENT_DATE())
) w
ON f.warehouse_id = CAST(w.warehouse_id AS INT)
   AND f.vivo_product_id = w.vivo_retail_id
WHERE f.country = 'CO'
  AND f.date BETWEEN CURRENT_DATE AND CURRENT_DATE + INTERVAL '14 DAYS'
  AND UPPER(w.cat2_name) = 'FRUTAS Y VERDURAS';
"""

In [4]:
#Ejecutar query de forecast en snowflake
df = pd.read_sql(query_forecast, snowflake_connection)

C:\Users\juane\AppData\Local\Temp\ipykernel_17976\4184599388.py:2: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query_forecast, snowflake_connection)


In [5]:
for variable in df['PROVIDER_NAME'].unique():
    temp_df = df[df['PROVIDER_NAME'] == variable]
    file_name = f"C:/Users/juane/Downloads/FORECAST_{variable}.csv"
    temp_df.to_csv(file_name, index=False)